# Test Radiology Vision Plugin (GPT-4o)

This notebook demonstrates testing the radiology_vision.py plugin using GPT-4o Vision.
No separate model deployment needed - uses existing GPT-4o deployment.

In [ ]:
import os
import sys

CUR_DIR = os.path.abspath("")
PROJECT_ROOT = os.path.dirname(os.path.dirname(CUR_DIR))
sys.path.append(os.path.join(PROJECT_ROOT, "src"))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
from dotenv import load_dotenv

env_path = os.path.join(PROJECT_ROOT, ".azure", os.getenv("AZURE_ENV_NAME", "dev"), ".env")
load_dotenv(dotenv_path=env_path)
print(f"Loaded environment from: {env_path}")

In [ ]:
# Test parameters
patient_id = "patient_4"
filename = "x-ray.png"  # or "ct_scan.png"
indication = "Chest pain"

print(f"Test Configuration:")
print(f"  Patient ID: {patient_id}")
print(f"  Filename: {filename}")
print(f"  Indication: {indication}")

In [ ]:
from config import load_agent_config
from data_models.chat_context import ChatContext
from data_models.plugin_configuration import PluginConfiguration
from scenarios.default.tools.radiology_vision import create_plugin
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai.services.azure_chat_completion import AzureChatCompletion
from app import create_app_context

# Setup kernel with GPT-4o
kernel = Kernel()
kernel.add_service(
    AzureChatCompletion(
        service_id="default",
        deployment_name=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
        endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        api_key=os.environ.get("AZURE_OPENAI_API_KEY"),
        api_version="2024-12-01-preview"
    )
)

print("✅ Kernel created with GPT-4o service")

In [ ]:
# Create app context and chat context
app_ctx = create_app_context()
chat_ctx = ChatContext("test-radiology-vision-" + os.urandom(4).hex())
chat_ctx.patient_id = patient_id

print(f"✅ App context created")
print(f"✅ Chat context created: {chat_ctx.conversation_id}")

In [ ]:
# Create plugin
agent_config = load_agent_config("default")
radiology_agent_config = next((config for config in agent_config if config["name"] == "Radiology"), None)

if not radiology_agent_config:
    raise ValueError("Radiology agent configuration not found in agents.yaml")

plugin_config = PluginConfiguration(
    kernel=kernel,
    chat_ctx=chat_ctx,
    agent_config=radiology_agent_config,
    data_access=app_ctx.data_access,
    azureml_token_provider=app_ctx.azureml_token_provider
)

radiology_vision_plugin = create_plugin(plugin_config)
print("✅ Radiology Vision Plugin created")

In [ ]:
# Test the plugin
print("🔍 Analyzing radiology image with GPT-4o Vision...\n")
print("=" * 80)

findings = await radiology_vision_plugin.analyze_radiology_image(
    patient_id=patient_id,
    filename=filename,
    indication=indication
)

print("\n" + "=" * 80)
print("RADIOLOGY FINDINGS (GPT-4o Vision)")
print("=" * 80)
print(findings)
print("=" * 80)

## Test with Different Images

Uncomment and modify the cell below to test with different images:

In [ ]:
# # Test with CT scan
# ct_findings = await radiology_vision_plugin.analyze_radiology_image(
#     patient_id="patient_4",
#     filename="ct_scan.png",
#     indication="Follow-up for lung nodule"
# )
# 
# print("\n" + "=" * 80)
# print("CT SCAN FINDINGS (GPT-4o Vision)")
# print("=" * 80)
# print(ct_findings)
# print("=" * 80)